# LLM-Based Narrative Block Extraction for DMPBridge

## Objective

The goal of this experiment is to evaluate whether Llama 3.1 8B via Ollama can identify and extract the narrative structure of Data Management Plans (DMPs).

In this phase, the LLM is used only to label PDFPlumber-extracted text blocks. It does not generate final DMP JSON, RDA JSON, or DMPTool narrative JSON.

---

## Workflow

PDF → PDFPlumber Extraction → PDFPlumber Extracted Blocks JSON → Llama 3.1 8B → Structured Blocks

---

## Input

The input is the JSON file generated by PDFPlumber and stored in:

`data/pdfplumber_extracted_blocks/`

Each block contains extracted text and may include layout information such as page number, font size, and bold formatting.

---

## Structured Block Labels

| Label | Description |
|---|---|
| `document_title` | Main title of the DMP |
| `section` | Existing major DMP heading |
| `subsection` | Existing prompt, question, or subheading |
| `content` | Narrative body text, instructions, guidance, or answers |

---

## Extraction Principle

The LLM performs extraction only. It should preserve original wording and ordering, use existing PDFPlumber blocks, and avoid creating new headings or summarizing content into headings.

If the model is unsure, it should label the text as `content`.

---

## Example Input Block

```json
{
  "text": "Element 1: Data Type:",
  "page": 1,
  "font_size": 12,
  "is_bold": true
}
```
## Example Output Block
```json

  {
    "label": "document_title",
    "text": "DATA MANAGEMENT AND SHARING PLAN"
  },
  {
    "label": "section",
    "text": "Element 1: Data Type:"
  },
  {
    "label": "subsection",
    "text": "A. Types and amount of scientific data expected to be generated in the project:"
  },
  {
    "label": "content",
    "text": "This project will generate..."
  }


In [1]:
from pathlib import Path
import json
import importlib

from dmpbridge.llm.llama_client import load_llama
import dmpbridge.llm.llm_narrative_blocks_new as lnb
from dmpbridge.processing.text_cleaner import clean_repeated_words


# Reload module to avoid old cached notebook version

importlib.reload(lnb)

generate_structured_blocks_with_llm = lnb.generate_structured_blocks_with_llm
save_blocks = lnb.save_blocks


# Project root

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

print("Project root:", project_root)


# Input / Output directories

pdfplumber_blocks_dir = (
    project_root
    / "data"
    / "pdfplumber_extracted_blocks"
)

blocks_output_dir = (
    project_root
    / "data"
    / "llama_structured_blocks"
)

blocks_output_dir.mkdir(
    parents=True,
    exist_ok=True,
)


# Load Llama once

llm = load_llama(
    model_name="llama3.1:8b",
    temperature=0,
)

print("Llama loaded successfully.")


# Process all PDFPlumber block JSON files

pdfplumber_block_files = sorted(
    pdfplumber_blocks_dir.glob("*.json")
)

print(f"\nFound {len(pdfplumber_block_files)} PDFPlumber block files")

for block_path in pdfplumber_block_files:

    sample_name = block_path.stem

    print("\n" + "=" * 80)
    print(f"Processing: {sample_name}")
    print("=" * 80)

    try:
        # Load PDFPlumber blocks

        with open(block_path, "r", encoding="utf-8") as f:
            pdfplumber_blocks = json.load(f)

        # Clean repeated words in each block before LLM extraction

        cleaned_pdfplumber_blocks = []

        for block in pdfplumber_blocks:
            cleaned_block = dict(block)

            cleaned_text = clean_repeated_words(
                str(block.get("text", ""))
            ).strip()

            if not cleaned_text:
                continue

            cleaned_block["text"] = cleaned_text
            cleaned_pdfplumber_blocks.append(cleaned_block)

        # Generate structured blocks from cleaned PDFPlumber blocks

        structured_blocks = generate_structured_blocks_with_llm(
            llm=llm,
            pdf_blocks=cleaned_pdfplumber_blocks,
        )

        # Save structured blocks only

        blocks_output_path = (
            blocks_output_dir
            / f"{sample_name}_llama_blocks.json"
        )

        save_blocks(
            blocks=structured_blocks,
            output_path=blocks_output_path,
        )

        print(
            f"Saved {len(structured_blocks)} structured blocks: "
            f"{blocks_output_path.name}"
        )

    except Exception as e:
        print(f"ERROR processing {sample_name}")
        print(e)

print("\nFinished processing all PDFPlumber block files.")

Project root: c:\Users\Nahid\dmpbridge
Llama loaded successfully.

Found 10 PDFPlumber block files

Processing: sample1
Saved 51 structured blocks: sample1_llama_blocks.json

Processing: sample10
Saved 15 structured blocks: sample10_llama_blocks.json

Processing: sample2
Saved 41 structured blocks: sample2_llama_blocks.json

Processing: sample3
Saved 12 structured blocks: sample3_llama_blocks.json

Processing: sample4
Saved 56 structured blocks: sample4_llama_blocks.json

Processing: sample5
Saved 31 structured blocks: sample5_llama_blocks.json

Processing: sample6
Saved 11 structured blocks: sample6_llama_blocks.json

Processing: sample7
Saved 9 structured blocks: sample7_llama_blocks.json

Processing: sample8
Saved 33 structured blocks: sample8_llama_blocks.json

Processing: sample9
Saved 12 structured blocks: sample9_llama_blocks.json

Finished processing all PDFPlumber block files.
